In [1]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

import evaluate

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [3]:
# Load NER dataset
dataset = load_dataset("flaitenberger/wnut_17")

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})

In [5]:
dataset['train'][1]

{'id': '1',
 'tokens': ['From',
  'Green',
  'Newsfeed',
  ':',
  'AHFA',
  'extends',
  'deadline',
  'for',
  'Sage',
  'Award',
  'to',
  'Nov',
  '.',
  '5',
  'http://tinyurl.com/24agj38'],
 'ner_tags': [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [6]:
label_names = dataset['train'].features['ner_tags'].feature.names
label_names

['O',
 'B-corporation',
 'I-corporation',
 'B-creative-work',
 'I-creative-work',
 'B-group',
 'I-group',
 'B-location',
 'I-location',
 'B-person',
 'I-person',
 'B-product',
 'I-product']

In [7]:
label2id = {
    label:idx
    for idx,label in enumerate(label_names)
}

id2label = {
    idx:label
    for idx,label in enumerate(label_names)
}

In [8]:
print(label2id)
print()
print(id2label)

{'O': 0, 'B-corporation': 1, 'I-corporation': 2, 'B-creative-work': 3, 'I-creative-work': 4, 'B-group': 5, 'I-group': 6, 'B-location': 7, 'I-location': 8, 'B-person': 9, 'I-person': 10, 'B-product': 11, 'I-product': 12}

{0: 'O', 1: 'B-corporation', 2: 'I-corporation', 3: 'B-creative-work', 4: 'I-creative-work', 5: 'B-group', 6: 'I-group', 7: 'B-location', 8: 'I-location', 9: 'B-person', 10: 'I-person', 11: 'B-product', 12: 'I-product'}


In [9]:
# Load Tokenizer 

model_checkpoint = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [10]:
example = dataset['train'][0]

for nertags,tokens in zip(example['ner_tags'],example['tokens']):
    print(f"{tokens:10} ---> {nertags}-->{id2label[nertags]}")

@paulwalk  ---> 0-->O
It         ---> 0-->O
's         ---> 0-->O
the        ---> 0-->O
view       ---> 0-->O
from       ---> 0-->O
where      ---> 0-->O
I          ---> 0-->O
'm         ---> 0-->O
living     ---> 0-->O
for        ---> 0-->O
two        ---> 0-->O
weeks      ---> 0-->O
.          ---> 0-->O
Empire     ---> 7-->B-location
State      ---> 8-->I-location
Building   ---> 8-->I-location
=          ---> 0-->O
ESB        ---> 7-->B-location
.          ---> 0-->O
Pretty     ---> 0-->O
bad        ---> 0-->O
storm      ---> 0-->O
here       ---> 0-->O
last       ---> 0-->O
evening    ---> 0-->O
.          ---> 0-->O


In [11]:
tokenized = tokenizer(
    example['tokens'],
    is_split_into_words= True
)

In [12]:
tokens = tokenizer.convert_ids_to_tokens(
    tokenized["input_ids"]
)

tokens

['[CLS]',
 '@',
 'paul',
 '##walk',
 'it',
 "'",
 's',
 'the',
 'view',
 'from',
 'where',
 'i',
 "'",
 'm',
 'living',
 'for',
 'two',
 'weeks',
 '.',
 'empire',
 'state',
 'building',
 '=',
 'es',
 '##b',
 '.',
 'pretty',
 'bad',
 'storm',
 'here',
 'last',
 'evening',
 '.',
 '[SEP]']

In [13]:
# now the words aare again splitted further 
# for further split on same we need same NER tag 
# like for @Paulwalk--> @  paul walk  we again need O O O(0 0 0) 

word_ids = tokenized.word_ids()

for tokens,nertags in zip(tokens,word_ids):
    print(f"{tokens:10} --> {nertags}")

[CLS]      --> None
@          --> 0
paul       --> 0
##walk     --> 0
it         --> 1
'          --> 2
s          --> 2
the        --> 3
view       --> 4
from       --> 5
where      --> 6
i          --> 7
'          --> 8
m          --> 8
living     --> 9
for        --> 10
two        --> 11
weeks      --> 12
.          --> 13
empire     --> 14
state      --> 15
building   --> 16
=          --> 17
es         --> 18
##b        --> 18
.          --> 19
pretty     --> 20
bad        --> 21
storm      --> 22
here       --> 23
last       --> 24
evening    --> 25
.          --> 26
[SEP]      --> None


In [14]:
# Implement label alignment

def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
      )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            # Special token
            if word_idx is None:
                label_ids.append(-100)

            # First subword of a word
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])

            # Remaining subwords
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [15]:
test = tokenize_and_align_labels({
    "tokens": [example["tokens"]],
    "ner_tags": [example["ner_tags"]]
})

In [16]:
aligned_labels = test["labels"][0]

for token, label_id in zip(
    tokenizer.convert_ids_to_tokens(test["input_ids"][0]),
    aligned_labels
):
    print(f"{token:15} -> {label_id}")

[CLS]           -> -100
@               -> 0
paul            -> -100
##walk          -> -100
it              -> 0
'               -> 0
s               -> -100
the             -> 0
view            -> 0
from            -> 0
where           -> 0
i               -> 0
'               -> 0
m               -> -100
living          -> 0
for             -> 0
two             -> 0
weeks           -> 0
.               -> 0
empire          -> 7
state           -> 8
building        -> 8
=               -> 0
es              -> 7
##b             -> -100
.               -> 0
pretty          -> 0
bad             -> 0
storm           -> 0
here            -> 0
last            -> 0
evening         -> 0
.               -> 0
[SEP]           -> -100


In [17]:
# applying to entire dataset 

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

In [18]:
tokenized_dataset["train"]

Dataset({
    features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3394
})

In [19]:
tokenized_dataset['train'][0]

{'id': '0',
 'tokens': ['@paulwalk',
  'It',
  "'s",
  'the',
  'view',
  'from',
  'where',
  'I',
  "'m",
  'living',
  'for',
  'two',
  'weeks',
  '.',
  'Empire',
  'State',
  'Building',
  '=',
  'ESB',
  '.',
  'Pretty',
  'bad',
  'storm',
  'here',
  'last',
  'evening',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  8,
  8,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'input_ids': [101,
  1030,
  2703,
  17122,
  2009,
  1005,
  1055,
  1996,
  3193,
  2013,
  2073,
  1045,
  1005,
  1049,
  2542,
  2005,
  2048,
  3134,
  1012,
  3400,
  2110,
  2311,
  1027,
  9686,
  2497,
  1012,
  3492,
  2919,
  4040,
  2182,
  2197,
  3944,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [20]:
# dynamic padding 

data_collator = DataCollatorForTokenClassification(
    tokenizer = tokenizer
)

In [21]:
# loading the model

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2656.47it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_predictions):

    predictions, labels = eval_predictions

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [
            label_names[p]
            for p, l in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            label_names[l]
            for p, l in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [23]:
training_args = TrainingArguments(
    output_dir='./ner-distilbert',

    learning_rate = 2e-5,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy='epoch',
    save_strategy='epoch',

    load_best_model_at_end=True,

    metric_for_best_model= 'f1',

    report_to='none'
)

In [24]:
trainer = Trainer(
    model= model,
    args= training_args,

    train_dataset= tokenized_dataset['train'],
    eval_dataset= tokenized_dataset['validation'],

    processing_class= tokenizer,

    data_collator= data_collator,

    compute_metrics= compute_metrics
)

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.271360,0.604010,0.288278,0.390283,0.938585
2,No log,0.246829,0.685832,0.399522,0.504913,0.945324
3,No log,0.237238,0.673432,0.436603,0.529753,0.947867


Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.44it/s]
Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.36it/s]


TrainOutput(global_step=321, training_loss=0.22255031787717824, metrics={'train_runtime': 103.4452, 'train_samples_per_second': 98.429, 'train_steps_per_second': 3.103, 'total_flos': 147180893953860.0, 'train_loss': 0.22255031787717824, 'epoch': 3.0})

In [26]:
result = trainer.evaluate()

print(result)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
No log,0.237238,3,0.673432,0.436603,0.529753,0.947867


{'eval_loss': 0.23723818361759186, 'eval_precision': 0.6734317343173432, 'eval_recall': 0.4366028708133971, 'eval_f1': 0.529753265602322, 'eval_accuracy': 0.9478669972661962}


In [27]:
# Testing Manually 

text = """
Apple CEO Tim Cook visited New York last Monday.
"""

In [28]:
inputs = tokenizer(
    text,
    return_tensors="pt"
)

In [29]:
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

In [30]:
model.eval()

with torch.no_grad():
    outputs = model(**inputs)

In [31]:
logits = outputs.logits

logits.shape

torch.Size([1, 12, 13])

In [32]:
predictions = torch.argmax(
    logits,
    dim=-1
)

In [33]:
predicted_labels = [
    id2label[pred.item()]
    for pred in predictions[0]
]

In [34]:
input_tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

In [35]:
for token, label in zip(input_tokens, predicted_labels):
    print(f"{token:15} -> {label}")

[CLS]           -> O
apple           -> B-corporation
ceo             -> O
tim             -> B-person
cook            -> I-person
visited         -> O
new             -> B-location
york            -> B-location
last            -> O
monday          -> O
.               -> O
[SEP]           -> O


In [36]:
# use the pipeline

from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

In [37]:
text = "Apple CEO Tim Cook visited New York last Monday."

entities = ner_pipeline(text)

entities

[{'entity_group': 'corporation',
  'score': np.float32(0.21683173),
  'word': 'apple',
  'start': 0,
  'end': 5},
 {'entity_group': 'person',
  'score': np.float32(0.81087023),
  'word': 'tim cook',
  'start': 10,
  'end': 18},
 {'entity_group': 'location',
  'score': np.float32(0.6474561),
  'word': 'new',
  'start': 27,
  'end': 30},
 {'entity_group': 'location',
  'score': np.float32(0.54640645),
  'word': 'york',
  'start': 31,
  'end': 35}]